<a href="https://colab.research.google.com/github/RiccoFlores/Python/blob/master/WalkIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# 1. Upload the file
df_GC = pd.read_csv('GuestCenter.csv')
df_GC.head()

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Visit Date,Visit Time,Seated Date,Finished Date,Size,Table,Source
0,21/11/2026,15:00,21/11/2026 14:45,NaN,2,32,Your Network
1,21/11/2026,15:00,21/11/2026 14:45,NaN,2,23,Your Network
2,21/11/2026,15:00,21/11/2026 14:49,NaN,6,24,Phone/In-house
3,21/11/2026,14:49,21/11/2026 14:49,NaN,5,40,Walk-in
4,21/11/2026,14:49,21/11/2026 14:49,NaN,2,42,Walk-in


In [24]:
# 2. Numeric Summary
df_GC.describe(include="all")


,Visit Date,Visit Time,Seated Date,Finished Date,Size,Table,Source
count,13919,13919,13919,9548,13919.000000,13919,13919
unique,320,354,10642,4898,NaN,145,4
top,25/07/2026,13:00,28/03/2026 15:14,17/07/2026 17:29,NaN,34,Walk-in
freq,112,1561,17,35,NaN,569,6644
mean,NaN,NaN,NaN,NaN,2.706947,NaN,NaN
std,NaN,NaN,NaN,NaN,1.906788,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,2.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,2.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,3.000000,NaN,NaN


In [25]:
# 2.1 Data Understanding, let's clarify our categories in column Source
print("Booking Sources:\n", df_GC['Source'].value_counts(dropna=False))


Booking Sources:
 Source
Walk-in              6644
Your Network         4489
OpenTable Network    1737
Phone/In-house       1049
Name: count, dtype: int64


In [28]:
# 2.2 Data Understanding, let's check if we are with an actual DATE format
df_GC.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13919 entries, 0 to 13918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Visit Date     13919 non-null  object
 1   Visit Time     13919 non-null  object
 2   Seated Date    13919 non-null  object
 3   Finished Date  9548 non-null   object
 4   Size           13919 non-null  int64 
 5   Table          13919 non-null  object
 6   Source         13919 non-null  object
dtypes: int64(1), object(6)
memory usage: 761.3+ KB


In [32]:
# 3. Parse Seated Date and Finished Date (dayfirst=True handles DD/MM/YYYY safely)
df_GC['Seated Date'] = pd.to_datetime(df_GC['Seated Date'], dayfirst=True, errors='coerce')
df_GC['Finished Date'] = pd.to_datetime(df_GC['Finished Date'], dayfirst=True, errors='coerce')

# 3.1 Extract pure date string from Visit Date and pure time string from Visit Time
visit_date_str = pd.to_datetime(df_GC['Visit Date'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')

In [34]:
# 3.1 Handle Visit Time whether Excel stored it as datetime.time, full datetime, or text
def extract_time_str(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    # If Excel attached a dummy date like '1899-12-30 13:30:00', keep only the time portion
    if ' ' in val_str:
        return val_str.split(' ')[-1]
    return val_str

visit_time_str = df_GC['Visit Time'].apply(extract_time_str)

In [35]:
# 3.2 Merge into 'Booking Date and Time'
df_GC['Booking Date and Time'] = pd.to_datetime(
    visit_date_str + ' ' + visit_time_str,
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

# Fallback if Visit Time had HH:MM without seconds
mask_nat = df_GC['Booking Date and Time'].isna() & visit_date_str.notna() & visit_time_str.notna()
df_GC.loc[mask_nat, 'Booking Date and Time'] = pd.to_datetime(
    visit_date_str[mask_nat] + ' ' + visit_time_str[mask_nat],
    errors='coerce'
)

# Verify the new column and null counts
display(df_GC[['Visit Date', 'Visit Time', 'Booking Date and Time', 'Seated Date', 'Finished Date']].head())
print("\nNull counts after parsing:")
print(df_GC[['Booking Date and Time', 'Seated Date', 'Finished Date']].isna().sum())

,Visit Date,Visit Time,Booking Date and Time,Seated Date,Finished Date
0,21/11/2026,15:00,2026-11-21 15:00:00,2026-11-21 14:45:00,NaT
1,21/11/2026,15:00,2026-11-21 15:00:00,2026-11-21 14:45:00,NaT
2,21/11/2026,15:00,2026-11-21 15:00:00,2026-11-21 14:49:00,NaT
3,21/11/2026,14:49,2026-11-21 14:49:00,2026-11-21 14:49:00,NaT
4,21/11/2026,14:49,2026-11-21 14:49:00,2026-11-21 14:49:00,NaT



Null counts after parsing:
Booking Date and Time       0
Seated Date                 0
Finished Date            4371
dtype: int64


In [36]:
df_GC.head()

,Visit Date,Visit Time,Seated Date,Finished Date,Size,Table,Source,Booking Date and Time
0,21/11/2026,15:00,2026-11-21 14:45:00,NaT,2,32,Your Network,2026-11-21 15:00:00
1,21/11/2026,15:00,2026-11-21 14:45:00,NaT,2,23,Your Network,2026-11-21 15:00:00
2,21/11/2026,15:00,2026-11-21 14:49:00,NaT,6,24,Phone/In-house,2026-11-21 15:00:00
3,21/11/2026,14:49,2026-11-21 14:49:00,NaT,5,40,Walk-in,2026-11-21 14:49:00
4,21/11/2026,14:49,2026-11-21 14:49:00,NaT,2,42,Walk-in,2026-11-21 14:49:00


In [44]:
# 4. Some data cleaning
# Remove the original 'Visit Date' and 'Visit Time' columns
df_GC = df_GC.drop(columns=['Visit Date', 'Visit Time'], errors='ignore')

# 4.1 Move 'Booking Date and Time' to the first position
cols = ['Booking Date and Time'] + [col for col in df_GC.columns if col != 'Booking Date and Time']
df_GC = df_GC[cols]

print("Updated columns:", df_GC.columns.tolist())
display(df_GC.head())

Updated columns: ['Booking Date and Time', 'Seated Date', 'Finished Date', 'Size', 'Table', 'Source']


,Booking Date and Time,Seated Date,Finished Date,Size,Table,Source
0,2026-11-21 15:00:00,2026-11-21 14:45:00,NaT,2,32,Your Network
1,2026-11-21 15:00:00,2026-11-21 14:45:00,NaT,2,23,Your Network
2,2026-11-21 15:00:00,2026-11-21 14:49:00,NaT,6,24,Phone/In-house
3,2026-11-21 14:49:00,2026-11-21 14:49:00,NaT,5,40,Walk-in
4,2026-11-21 14:49:00,2026-11-21 14:49:00,NaT,2,42,Walk-in


In [45]:
# 5. Calculate durations in minutes for valid, non-null rows
df_GC['wait_minutes'] = (df_GC['Seated Date'] - df_GC['Booking Date and Time']).dt.total_seconds() / 60
df_GC['dwell_minutes'] = (df_GC['Finished Date'] - df_GC['Seated Date']).dt.total_seconds() / 60

# Filter valid dwell times (e.g., between 10 mins and 360 mins / 6 hours) to avoid unclosed-till outliers
valid_dwell = df_GC[(df_GC['dwell_minutes'] >= 10) & (df_GC['dwell_minutes'] <= 360)]

mean_dwell = valid_dwell['dwell_minutes'].mean()
median_dwell = valid_dwell['dwell_minutes'].median()

print(f"Valid completed sittings analyzed: {len(valid_dwell)}")
print(f"Average table time in use (Mean):   {mean_dwell:.1f} minutes ({mean_dwell/60:.2f} hours)")
print(f"Median table time in use (Median):  {median_dwell:.1f} minutes ({median_dwell/60:.2f} hours)")

# Check how dwell time varies by party Size (optional check)
print("\nAverage Dwell Time (mins) by Party Size:")
display(valid_dwell.groupby('Size')['dwell_minutes'].agg(['count', 'mean', 'median']).round(1))

Valid completed sittings analyzed: 8880
Average table time in use (Mean):   97.2 minutes (1.62 hours)
Median table time in use (Median):  86.0 minutes (1.43 hours)

Average Dwell Time (mins) by Party Size:


,count,mean,median
Size,,,
1,830,75.5,62.0
2,4984,94.5,83.0
3,1408,106.6,98.0
4,969,105.1,97.0
5,323,107.1,102.0
6,174,112.2,105.0
7,49,119.9,109.0
8,38,131.8,122.0
9,17,117.4,106.0
